# Figure 3: Joint cell–gene representation

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


## Joint embedding


In [ ]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad("../../data/kang_2018.h5ad")

# Match the exported IDs: training can remove unexpressed genes.
adata = adata[
    np.load("../../results/kang/scLDM/scLDM_cell_names.npy"),
    np.load("../../results/kang/scLDM/scLDM_gene_names.npy"),
].copy()

z_cells = np.load('../../results/kang/scLDM/scLDM_cell_latent.npy')
z_genes = np.load('../../results/kang/scLDM/scLDM_gene_latent.npy')

if z_cells.shape[1] != z_genes.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells.shape}, genes={z_genes.shape}")

adata.obsm['scLDM'] = z_cells
adata.varm['scLDM'] = z_genes

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

# ----- 1.  Build a placeholder expression matrix for the genes -----
# Gene rows have zero-filled expression values; their embeddings are stored separately.
n_cells, n_genes = adata.n_obs, adata.n_vars
if sp.issparse(adata.X):
    placeholder_X = sp.csr_matrix((n_genes, n_genes), dtype=adata.X.dtype)
else:
    placeholder_X = np.zeros((n_genes, n_genes), dtype=adata.X.dtype)

# ----- 2.  Stack the real cells with the pseudo-cells (genes) -----
X_combined = (
    sp.vstack([adata.X, placeholder_X], format="csr")
    if sp.issparse(adata.X)
    else np.vstack([adata.X, placeholder_X])
)

# ----- 3.  Create an .obs that labels each row as cell / gene -----
obs_combined = pd.concat(
    [
        adata.obs.assign(entity="cell"),               # keep existing cell metadata
        pd.DataFrame({"entity": "gene"}, index=adata.var_names)  # one row per gene
    ]
)

# ----- 4.  Assemble the new AnnData object -----
adata_combo = sc.AnnData(
    X=X_combined,
    obs=obs_combined,
    var=adata.var.copy()            # keep original gene metadata as .var
)

# ----- 5.  Concatenate the embeddings and store in .obsm -----

adata_combo.obsm["scLDM"] = np.vstack([
    adata.obsm["scLDM"],      # cells (n_cells × dim)
    adata.varm["scLDM"]       # genes (n_genes × dim)
])


In [ ]:
import scanpy as sc
sc.pp.neighbors(adata_combo, use_rep="scLDM")
sc.tl.umap(adata_combo, random_state=42, key_added="umap_scLDM")

In [ ]:
from scanpy.plotting import palettes as scpal
from matplotlib.colors import to_hex
import matplotlib.pyplot as plt


celltypes = adata_combo.obs["cell_type"]
categories = pd.Categorical(celltypes).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette_celltypes = dict(zip(categories, colors))



stim_groups = adata_combo.obs["label"]
categories = pd.Categorical(stim_groups).categories

n_cat = len(categories)

if n_cat <= 20:
    colors = scpal.default_20[:n_cat]
elif n_cat <= 28:
    colors = scpal.default_28[:n_cat]
elif n_cat <= 102:
    colors = scpal.default_102[:n_cat]
else:
    cmap = plt.get_cmap("gist_ncar", n_cat)
    colors = [to_hex(cmap(i)) for i in range(n_cat)]

palette_stim_groups = dict(zip(categories, colors))



In [ ]:
adata_combo

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import numpy as np
import pandas as pd
from adjustText import adjust_text
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

})

# -----------------------------
# Data
# -----------------------------
umap = np.asarray(adata_combo.obsm["umap_scLDM"], dtype=float)

cell_mask = (adata_combo.obs["entity"] == "cell").to_numpy()
gene_mask = (adata_combo.obs["entity"] == "gene").to_numpy()

genes_xy = umap[gene_mask]
gene_names = adata_combo.obs_names[gene_mask].to_numpy()  # should match adata_combo.var_names


# -----------------------------
# Helpers
# -----------------------------
def build_palette(categories):
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return dict(zip(categories, colors))

# -----------------------------
# Plot
# -----------------------------
labels = ["cell_type", "label"]
rng = np.random.default_rng(42)  # reproducible random draw order

for label in labels:
    # categories from CELLS only
    cell_labels_series = adata_combo.obs.loc[cell_mask, label].astype(str)
    categories = list(pd.Categorical(cell_labels_series).categories)

    if label == "cell_type" and "palette_celltypes" in globals():
        palette = palette_celltypes
    else:
        palette = build_palette(categories)

    fig, ax = plt.subplots(figsize=(3.5, 3.5))

    # Randomize category plotting order (reduces systematic overplot bias)
    categories_plot = categories.copy()
    rng.shuffle(categories_plot)

    label_all = adata_combo.obs[label].astype(str).to_numpy()

    # Cells by category
    for ct in categories_plot:
        idx = cell_mask & (label_all == ct)
        point_ids = np.flatnonzero(idx)
        if point_ids.size == 0:
            continue
        point_ids = rng.permutation(point_ids)  # random within category
        ax.scatter(
            umap[point_ids, 0],
            umap[point_ids, 1],
            s=1,
            c=[palette[ct]],
            alpha=1,
            linewidths=0,
            label=ct,
            rasterized=True

        )

    # Genes colored by n_counts
    gene_sc = ax.scatter(
        genes_xy[:, 0],
        genes_xy[:, 1],
        s=1,
        c="#7A1E1E",
        #cmap="viridis",
        #vmin=vmin,
        #vmax=vmax,
        alpha=1,
        linewidths=0,
        label="genes",
        rasterized=True
    )

    #cbar = fig.colorbar(gene_sc, ax=ax, fraction=0.03, pad=0.01)
    #cbar.set_label("log1p(n_counts)", fontsize=6)
    #cbar.ax.tick_params(labelsize=5, length=2)

    fig3a_marker_genes = {
        "CD4 T cells": [
            "LTB",
            "CCR7",
            "TRAC",
        ],
        "CD8 T cells": [
            "CD8A",
            "CD8B",
            "GZMK",
        ],
        "NK cells": [
            "NKG7",
            "GNLY",
            "KLRD1",
        ],
        "B cells": [
            "MS4A1",
            "CD79A",
            "CD79B",
        ],
        "CD14+ Monocytes": [
            "FCN1",
            "S100A8",
            "S100A9",
        ],
        "FCGR3A+ Monocytes": [
            "FCGR3A",
            "MS4A7",
            "LST1",
        ],
        "Dendritic cells": [
            "FCER1A",
            "CLEC10A",
            "CST3",
        ],
        "Other": [
            "CD74",
            "IL7R",
        ]
    }


    genes = [
        gene
        for genes in fig3a_marker_genes.values()
        for gene in genes
    ]

    if label == "cell_type":
        markers = genes
        selected_genes = np.array(markers)
        sel = np.isin(gene_names, selected_genes)
        genes_sel = genes_xy[sel]
        gene_names_sel = gene_names[sel]

        # Highlight labeled genes
        ax.scatter(
            genes_sel[:, 0],
            genes_sel[:, 1],
            s=6,
            facecolors="none",
            edgecolors="black",
            linewidths=0.2,
            alpha=1,
            zorder=5,
            rasterized=False,
        )

        # ax.scatter(
        #     genes_sel[:, 0],
        #     genes_sel[:, 1],
        #     s=4,
        #     c="#7A1E1E",
        #     linewidths=0,
        #     alpha=1,
        #     zorder=5,
        #     rasterized=False,
        # )

        texts = []
        for (x, y), name in zip(genes_sel, gene_names_sel):
            texts.append(
                ax.text(
                    x, y, name,
                    fontsize=5,
                    color="black",
                    alpha=1,
                    zorder=6,
                    bbox=dict(
                        boxstyle="round,pad=0.15",
                        facecolor="white",
                        edgecolor="none",
                        alpha=0.5
                    )
                )
            )

        adjust_text(
            texts,
            ax=ax,

            # stronger repulsion from other text labels
            expand_text=(1.8, 2.2),
            force_text=(0.8, 1.2),

            # stronger repulsion from the gene/cell points
            expand_points=(4.0, 4.0),
            force_points=(1.5, 2.0),

            # weaker pull back to original gene location
            force_pull=(0.005, 0.005),

            # allow more optimization
            lim=1500,

            # movement allowed in both directions
            only_move={
                "points": "xy",
                "text": "xy",
                "objects": "xy",
                "pull": "xy",
            },

            # always draw arrows, even for short displacements
            min_arrow_len=0,

            arrowprops=dict(
                arrowstyle="-",
                color="black",
                lw=0.25,
                alpha=0.7,
            ),
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(
        f"output/fig_3/kang_umap_{label}.svg",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600
    )
    plt.show()


In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def save_cluster_legend_pdf(
    categories,
    palette,
    output_pdf,
    ncol=1,
    fontsize=6,
    marker_size=4.0,
    columnspacing=0.8,
    handletextpad=0.4,
    labelspacing=0.35,
    borderpad=0.2,
):
    """
    Save a standalone legend-only PDF for cluster colors.

    Parameters
    ----------
    categories : list[str]
        Ordered category names.
    palette : dict
        Mapping {category: color}.
    output_pdf : str
        Output path ending in .pdf.
    ncol : int
        Number of legend columns.
    fontsize : float
        Legend text size in pt. Nature-style target: ~5–7 pt.
    marker_size : float
        Marker size in pt for legend keys.
    """

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    handles = [
        Line2D(
            [0], [0],
            linestyle="None",
            marker="o",
            markersize=marker_size,
            markerfacecolor=palette[cat],
            markeredgecolor=palette[cat],
            markeredgewidth=0.0,
            label=str(cat),
        )
        for cat in categories
    ]

    # Rough figure size estimate so the legend lays out predictably.
    n_items = len(categories)
    n_rows = math.ceil(n_items / ncol)
    fig_w = max(1.2, 1.15 * ncol + 0.55 * ncol)
    fig_h = max(0.35, 0.22 * n_rows + 0.18)

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.legend(
        handles=handles,
        labels=categories,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=0.8,
        handletextpad=handletextpad,
        columnspacing=columnspacing,
        labelspacing=labelspacing,
        borderpad=borderpad,
        markerscale=1.0,
    )

    fig.savefig(
        output_pdf,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )
    plt.close(fig)

In [ ]:
categories = list(adata_combo.obs["cell_type"].cat.categories)

save_cluster_legend_pdf(
    categories=categories,
    palette=palette_celltypes,
    output_pdf="output/fig_3/celltype_legend.svg",
    ncol=1,          # Number of legend columns
    fontsize=6,
    marker_size=4.0,
)

In [ ]:
val = palette_stim_groups.pop('ctrl', None)
if val is not None:
    palette_stim_groups['Control'] = val

val = palette_stim_groups.pop('stim', None)
if val is not None:
    palette_stim_groups['IFN-Beta'] = val

categories = ['Control', 'IFN-Beta']

save_cluster_legend_pdf(
    categories=categories,
    palette=palette_stim_groups,
    output_pdf="output/fig_3/stim_group_legend.svg",
    ncol=1,          # Number of legend columns
    fontsize=6,
    marker_size=4.0,
)

In [ ]:
palette_genes = {"Genes": "#7A1E1E"}
categories = ['Genes']

save_cluster_legend_pdf(
    categories=categories,
    palette=palette_genes,
    output_pdf="output/fig_3/genes_legend.svg",
    ncol=1,
    fontsize=6,
    marker_size=4.0,
)


In [ ]:
cell_identity_markers_strict = {
    "CD4 T cells": ["IL7R", "MAL", "LTB"],
    "CD14+ Monocytes": ["FCN1", "S100A8", "VCAN"],
    "B cells": ["MS4A1", "CD79A", "CD79B"],
    "NK cells": ["GNLY", "FGFBP2", "KLRF1"],
    "CD8 T cells": ["CD8A", "CD8B", "CTSW"],
    "FCGR3A+ Monocytes": ["FCGR3A", "MS4A7", "CDKN1C"],
    "Dendritic cells": ["FCER1A", "CD1C", "CLEC10A"],
    "Megakaryocytes": ["PPBP", "PF4", "GP9"],
}

In [ ]:
adata.obs['cell_type'].value_counts()

In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path


mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 5,
    "svg.fonttype": "none",   # keep text editable
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig3a_marker_genes = {
    "CD4 T cells": [
        "LTB",
        "CCR7",
        "TRAC",
    ],
    "CD8 T cells": [
        "CD8A",
        "CD8B",
        "GZMK",
    ],
    "NK cells": [
        "NKG7",
        "GNLY",
        "KLRD1",
    ],
    "B cells": [
        "MS4A1",
        "CD79A",
        "CD79B",
    ],
    "CD14+ Monocytes": [
        "FCN1",
        "S100A8",
        "S100A9",
    ],
    "FCGR3A+ Monocytes": [
        "FCGR3A",
        "MS4A7",
        "LST1",
    ],
    "Dendritic cells": [
        "FCER1A",
        "CLEC10A",
        "CST3",
    ],
    "Other": [
        "CD74",
        "IL7R",
    ]
}


genes = [
    gene
    for genes in fig3a_marker_genes.values()
    for gene in genes
]


# grid layout
n_cols = 4
n_rows = math.ceil(len(genes) / n_cols)

# figure size in inches
fig_w = 3.35
row_height = 0.22
fig_h = max(1.2, n_rows * row_height)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)
ax.axis("off")

for i, gene in enumerate(genes):
    row = n_rows - 1 - (i // n_cols)
    col = i % n_cols

    x = col + 0.5
    y = row + 0.5

    ax.text(
        x, y, gene,
        ha="center",
        va="center",
        fontsize=5,
        bbox=dict(
            facecolor="white",
            alpha=0.75,
            edgecolor="none",
            boxstyle="round,pad=0.18",
        ),
    )

outpath = Path("output/fig_3/gene_labels_grid.svg")
fig.savefig(
    outpath,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.02,
    transparent=True,
)
plt.show()
plt.close(fig)

print(f"Saved to: {outpath}")

## IFN-Beta gene scoring

In [ ]:
adata_combo

In [ ]:

ifn_beta_modules = {
    "Antigen_processing_immunoproteasome_core": ["NLRC5", "TAP1", "TAP2", "TAPBP", "B2M", "ERAP1", "PSMB8", "PSMB9", "PSMB10", "PSME1", "PSME2"],
    "OAS_RNaseL_core": ["OAS1", "OAS2", "OAS3", "OASL", "RNASEL"],
    "ISGylation_core": ["ISG15", "UBA7", "UBE2L6", "HERC5", "USP18"],
    "SREBP2_sterol_mevalonate_core": ["HMGCS1", "HMGCR", "MVK", "MVD", "IDI1", "FDPS", "FDFT1", "SQLE", "LSS", "DHCR7", "DHCR24"],
}



In [ ]:
import numpy as np
from itertools import chain

selected_genes = ["ISG15", "IFIT1", "MX1", "OAS1", "RSAD2", "USP18”, ”HMGCR", "HMGCS1", "SQLE”, ”IL1B", "NLRP3", "CASP1", "CXCL10"]

# Split rows
entity = adata_combo.obs["entity"].astype(str)
cell_mask = entity.eq("cell").to_numpy()
gene_mask = entity.eq("gene").to_numpy()

Z = adata_combo.obsm["scLDM"]
Z_cells = Z[cell_mask]                  # (n_cells, d)
Z_genes_all = Z[gene_mask]              # (n_genes, d)
gene_names_all = adata_combo.obs_names[gene_mask].to_numpy()

# Keep selected genes that exist among gene pseudo-cells
present = [g for g in selected_genes if g in set(gene_names_all)]
missing = sorted(set(selected_genes) - set(present))
print("Using:", present)
if missing:
    print("Missing:", missing)
if len(present) == 0:
    raise ValueError("No selected genes found in adata_combo gene rows.")

# Preserve selected_genes order
gene_pos = {g: i for i, g in enumerate(gene_names_all)}
idx = [gene_pos[g] for g in present]
Z_genes = Z_genes_all[idx]              # (n_selected, d)

# Euclidean distances: cells x selected_genes
D = np.linalg.norm(Z_cells[:, None, :] - Z_genes[None, :, :], axis=2)

# Aggregate scores per cell
mean_dist = D.mean(axis=1)
min_dist = D.min(axis=1)
centroid_dist = np.linalg.norm(Z_cells - Z_genes.mean(axis=0), axis=1)

# Store in adata_combo.obs (cells only; genes stay NaN)
for col, vals in [
    ("scLDM_mean_dist_selgenes", mean_dist),
    ("scLDM_min_dist_selgenes", min_dist),
    ("scLDM_centroid_dist_selgenes", centroid_dist),
]:
    adata_combo.obs[col] = np.nan
    adata_combo.obs.loc[cell_mask, col] = vals

# Optional: one column per selected gene distance
for j, g in enumerate(present):
    col = f"scLDM_dist_{g}"
    adata_combo.obs[col] = np.nan
    adata_combo.obs.loc[cell_mask, col] = D[:, j]


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import numpy as np
from adjustText import adjust_text
from matplotlib.colors import PowerNorm
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Extract coordinates
umap = adata_combo.obsm["umap_scLDM"]

cell_mask = (adata_combo.obs["entity"] == "cell").values
gene_mask = (adata_combo.obs["entity"] == "gene").values

cells = umap[cell_mask]
genes = umap[gene_mask]

ifn_beta_genes = ["ISG15", "IFIT1", "MX1", "OAS1", "RSAD2", "USP18”, ”HMGCR", "HMGCS1", "SQLE”, ”IL1B", "NLRP3", "CASP1", "CXCL10"]

labels = [f"scLDM_dist_{g}" for g in ifn_beta_genes]

for label in labels:
    try:
        vals = adata_combo.obs.loc[cell_mask, label].to_numpy(dtype=float)

        fig, ax = plt.subplots(figsize=(1.888, 1.60))

        vmin, vmax = np.nanpercentile(vals, [1, 95])

        norm = PowerNorm(gamma=0.5, vmin=vmin, vmax=vmax, clip=True)

        sm = mpl.cm.ScalarMappable(norm=norm, cmap="Blues_r")
        sm.set_array([])

        fig_cb, ax_cb = plt.subplots(figsize=(0.1, 2))
        cbar = fig_cb.colorbar(sm, cax=ax_cb)
        cbar.set_ticks([])
        cbar.set_label("Euclidean distance", fontsize=6, rotation=270, labelpad=8)
        fig_cb.savefig("output/fig_3/shared_distance_colorbar.svg", bbox_inches="tight", pad_inches=0)



        # Plot cells by cell type
        sca =ax.scatter(
            umap[cell_mask, 0],
            umap[cell_mask, 1],
            s=0.5,
            c=vals,
            cmap="Blues_r",
            norm=norm,
            alpha=1,
            #marker='.',
            linewidths=0,
            rasterized=True
        )

        # Overlay genes in one explicit color
        ax.scatter(
            genes[:, 0],
            genes[:, 1],
            s=0.2,
            c="grey",
            alpha=0.5,
            #marker='.',
            label="genes",
            linewidths=0,
            rasterized=True
        )

        ax.set_axis_off()

        gene_name = label.replace("scLDM_dist_", "")
        ax.set_title(gene_name, fontsize=5.5, pad=1)
        #ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=10)

        fig.savefig(f"output/fig_3/kang_umap_latent_{label}.svg", bbox_inches="tight", pad_inches=0, dpi=600)
        plt.show()
        #plt.close()
    except KeyError:
        print(f"Warning: {label} not found in adata_combo.obs; skipping.")




## Probability x Dist

In [ ]:
import numpy as np
import json

re_cells = np.load('../../results/kang/scLDM/scLDM_re_cell.npy')
re_genes = np.load('../../results/kang/scLDM/scLDM_re_gene.npy')


adata.obs['re_cells'] = re_cells
adata.var['re_genes'] = re_genes


path = "../../results/kang/scLDM/scLDM_params.json"

with open(path, "r") as f:
    params = json.load(f)

alpha = params["alpha"]


In [ ]:
selected_genes = [
    "IL7R",    # CD4 T
    "FCN1",    # CD14+ Monocytes
    "MS4A1",   # B cells
    "GNLY",    # NK cells
    "CTSW",    # CD8 T cells
    "FCGR3A",  # FCGR3A+ Monocytes
    "FCER1A",  # Dendritic cells
    "CD74",
    "NKG7"
]

In [ ]:
import numpy as np

Z_cells = np.asarray(adata.obsm["scLDM"])   # (n_cells, d)
Z_genes_all = np.asarray(adata.varm["scLDM"])
gene_names_all = adata.var_names

# keep requested genes that exist
present = [g for g in selected_genes if g in gene_names_all]
missing = sorted(set(selected_genes) - set(present))
print("Using:", present)
if missing:
    print("Missing:", missing)
if not present:
    raise ValueError("No selected genes found in adata.var_names.")

# preserve requested order
idx = gene_names_all.get_indexer(present)
Z_genes = Z_genes_all[idx, :]  # (n_selected, d)

# distances: cells x selected genes
D = np.linalg.norm(Z_cells[:, None, :] - Z_genes[None, :, :], axis=2)

# aggregate scores per cell
adata.obs["scLDM_mean_dist_selgenes"] = D.mean(axis=1)
adata.obs["scLDM_min_dist_selgenes"] = D.min(axis=1)
adata.obs["scLDM_centroid_dist_selgenes"] = np.linalg.norm(
    Z_cells - Z_genes.mean(axis=0), axis=1
)

# one column per selected gene
for j, g in enumerate(present):
    adata.obs[f"scLDM_dist_{g}"] = D[:, j]


In [ ]:
import numpy as np
from scipy.special import expit  # stable sigmoid

def reconstruct_pi_lambda_for_genes(adata, genes, alpha):
    # map genes -> var indices
    idx = adata.var_names.get_indexer(genes)
    keep = idx >= 0
    present_genes = [g for g, k in zip(genes, keep) if k]
    missing_genes = [g for g, k in zip(genes, keep) if not k]
    idx = idx[keep]
    if len(idx) == 0:
        raise ValueError("None of the requested genes were found in adata.var_names.")

    # model terms
    Z_cells = np.asarray(adata.obsm["scLDM"], dtype=np.float64)          # (n_cells, d)
    Z_genes = np.asarray(adata.varm["scLDM"][idx, :], dtype=np.float64)  # (k, d)
    re_cells = np.asarray(adata.obs["re_cells"], dtype=np.float64).reshape(-1, 1)  # (n_cells,1)
    re_genes = np.asarray(adata.var["re_genes"].values[idx], dtype=np.float64).reshape(1, -1)  # (1,k)

    # torch.cdist(..., p=2) equivalent
    dist = np.linalg.norm(Z_cells[:, None, :] - Z_genes[None, :, :], axis=2)  # (n_cells, k)

    # diff = re_mat - dist, with re_mat = re_cells + re_genes.T
    diff = (re_cells + re_genes) - dist

    # pi = sigmoid(alpha * diff)
    pi = expit(float(alpha) * diff)  # (n_cells, k)
    _lambda = np.exp(diff)  # (n_cells, k)

    return pi, _lambda, present_genes, missing_genes

# example
genes = selected_genes
alpha = params["alpha"]
pi, _lambda, present, missing = reconstruct_pi_lambda_for_genes(adata, genes, alpha)

# optional: save per-gene pi in obs
for j, g in enumerate(present):
    adata.obs[f"pi_{g}"] = pi[:, j]
    adata.obs[f"lambda_{g}"] = _lambda[:, j]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scanpy.plotting import palettes as scpal
from pathlib import Path
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 5,
    "axes.titlesize": 5,
    "axes.labelsize": 5,
    "xtick.labelsize": 5,
    "ytick.labelsize": 5,
    "svg.fonttype": "none",

})

genes = selected_genes

ct_col = "cell_type"
outdir = Path("output/fig_3/pi_vs_dist")
outdir.mkdir(parents=True, exist_ok=True)

cats = adata.obs[ct_col].cat.categories


for gene in genes:
    x_col = f"scLDM_dist_{gene}"
    y_col = f"pi_{gene}"

    if x_col not in adata.obs.columns or y_col not in adata.obs.columns:
        print(f"Skipping {gene}: missing {x_col} or {y_col}")
        continue

    x = adata.obs[x_col].to_numpy(dtype=float)
    y = adata.obs[y_col].to_numpy(dtype=float)
    ct = adata.obs[ct_col].astype(str).to_numpy()

    m = np.isfinite(x) & np.isfinite(y) & pd.notna(ct)
    x, y, ct = x[m], y[m], ct[m]

    fig, ax = plt.subplots(figsize=(1.68, 1.))
    for c in cats:
        idx = (ct == c)
        if np.any(idx):
            ax.scatter(x[idx], y[idx], s=0.5, alpha=0.7, linewidths=0, c=[palette_celltypes[c]], label=c, rasterized=True)
    
    ax.set_ylim(0,1)
    ax.set_title(gene, pad=2)
    #ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=2)

    plt.tight_layout()
    plt.savefig(outdir / f"pi_vs_dist_{gene}.svg", dpi=450, bbox_inches="tight")

    plt.show()
        # plt.close(fig)
